In [4]:
# Import the basic spark library
from pyspark.sql import SparkSession

# Create an entry point to the PySpark Application
spark = SparkSession.builder \
      .master("local") \
      .appName("ProjSMBUD") \
      .getOrCreate()
# master contains the URL of your remote spark instance or 'local'

In [2]:
# Upload list from a file
#rdd_2 = spark.sparkContext.textFile("") Don't needed? we use Dataframe

# Upload list from a file
#rdd_incollection = spark.sparkContext.textFile("incollectionFormatted.csv.txt")
#rdd_book = spark.sparkContext.textFile("bookFormatted.csv.txt")
#rdd_author = spark.sparkContext.textFile("authorFormatted.csv.txt")

#rdd_IwrittenbyA = spark.sparkContext.textFile("writtenBy-incollection-authorFormatted.csv")
#rdd_BeditedbyA = spark.sparkContext.textFile("editedBy-book-authorFormatted.csv")
#rdd_BcontainI = spark.sparkContext.textFile("contain-book-incollectionFormatted.csv")




#from pyspark.sql.types import StructType, StructField, StringType, FloatType, ArrayType

#Createe the schema using StructField(Name, Type, Nullable)  Incollection
#schema = StructType([ \
#    StructField("id", FloatType(), True), \
#    StructField("date", StringType(), True), \
#    StructField("url", StringType(), True), \
#    StructField("pages", StringType(), True), \
#    StructField("citationscore", FloatType(), True), \
#    StructField("title", StringType(), True) \
    
#])
#df_incollection = spark.createDataFrame(data = rdd_incollection, schema = schema)
#df_incollection.printSchema()
#df_incollection.show(truncate=False)

In [127]:
# Load DataFrames
#df_article = spark.read.option("header", True).option("delimiter", ",").csv("article.csv.txt")
df_incollection = spark.read.option("header", True).option("delimiter", ";").csv("incollectionFormatted.csv.txt")
df_book = spark.read.option("header", True).option("delimiter", ";").csv("bookFormatted1.csv.txt")
#df_www = spark.read.option("header", True).option("delimiter", ",").csv("www.csv.txt")
#df_proceedings = spark.read.option("header", True).option("delimiter", ",").csv("proceedings.csv.txt")
#df_mastersthesis = spark.read.option("header", True).option("delimiter", ",").csv("mastersthesis.csv.txt")
#df_inproceedings = spark.read.option("header", True).option("delimiter", ",").csv("inproceedings.csv.txt")
#df_articlen4j = spark.read.option("header", True).option("delimiter", ",").csv("export.csv.txt")
#df_records = spark.read.json("records.json")
df_author = spark.read.option("header", True).option("delimiter", ",").csv("authorFormatted.csv.txt")

#Load bridge tables (now they are in the entities)
#df_IwrittenbyA = spark.read.option("header", True).option("delimiter", ";").csv("writtenBy-inc-authorFormatted.csv.txt")
#df_BeditedbyA = spark.read.option("header", True).option("delimiter", ";").csv("editedBy-book-authorFormatted.csv.txt")
#df_BcontainI = spark.read.option("header", True).option("delimiter", ";").csv("contain-book-incollectionFormatted.csv.txt")
#df_IcitesI = spark.read.option("header", True).option("delimiter", ",").csv("citesFormatted.csv.txt")


# Print detected 
#df_IwrittenbyA.printSchema()
#df_IwrittenbyA.limit(5).show()

#df_BeditedbyA.printSchema()
#df_BeditedbyA.limit(5).show()

#df_BcontainI.printSchema()
#df_BcontainI.limit(5).show()

#df_IcitesI.printSchema()
#df_IcitesI.limit(5).show()

##df_book.show() #the representation has too columns so it is not good but it is good




# We use of datatypes from pyspark.sql.types
from pyspark.sql.types import StringType, DateType, FloatType, StructType, StructField, ArrayType
from pyspark.sql.functions import split, col, coalesce, array  #,lit, when  #for the arrays
#import pandas as pd
#import numpy as np



#df_incollection.printSchema()
#df_incollection.limit(5).select("id", 'author_array').show()
#we use the ids in string format!
df_incollection = df_incollection \
  .withColumn("id" ,
              df_incollection["id"]
              .cast(StringType()))   \
  .withColumn("title"  ,
              df_incollection["title"]
              .cast(StringType())) \
  .withColumn("date",
              df_incollection["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_incollection["url"]
              .cast(StringType())) \
  .withColumn("pages"  ,
              df_incollection["pages"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_incollection["citationscore"]
              .cast(FloatType())) \
#  .withColumn("author_array"  ,
#              df_incollection["author_array"]
#              .cast(ArrayType(StringType()))) \
#  .withColumn("cited_inc_array"  ,
#              df_incollection["cited_inc_array"]
#              .cast(ArrayType(StringType()))) \
#df_incollection = df_incollection.select(split(col("author_array"),",").alias("author_array"))
#df_incollection = df_incollection.selectExpr("cast(author_array as array(string)) author_array")

#df_incollection.printSchema()
#df_incollection.limit(5).show()

#df_book.printSchema()
#df_book.limit(5).select("id", "editor_array").show() #a test of uploading...
df_book_editors = df_book.select(split(col("editor_array"),",").alias("editors_array"))
#df_book_editors.withColumn('editor_array', coalesce(col('editor_array'), array(lit())))
#df_book_editors.withColumn("editor_array", coalesce('editor_array', array().cast("array<string>")))
#df_book_editors.na.fill("[]",["editor_array"])
#strs = [" "]
#df_book_editors.withColumn("editors_array", when(col("editors_array")==None , strs).otherwise(col("editors_array")))
#df_book_editors = df_book_editors.fillna(",")
#df_book_editors = df_book_editors.replace(np.nan, '')

df_book_editors.printSchema()
df_book_editors.show()
df_book.printSchema()
df_book = df_book \
  .withColumn("id" ,
              df_book["id"]
              .cast(StringType()))   \
  .withColumn("isbn"  ,
              df_book["isbn"]
              .cast(StringType())) \
  .withColumn("date",
              df_book["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_book["url"]
              .cast(StringType())) \
  .withColumn("publisher"  ,
              df_book["publisher"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_book["citationscore"]
              .cast(FloatType())) \
  .withColumn("title"  ,
              df_book["title"]
              .cast(StringType())) \
#  .withColumn("editor_array"  ,
#              df_book_editors["editors_array"]) \
#NON SI RIESCE AD INSERIRE LA COLONNA DI ARRAY DI EDITOR AL POSTO DI STRINGHE 
#DI EDITORS PERCHE HA VALORI NULLI E NON SI RIESCONO A TOGLIERE

df_book.printSchema()
df_book.show()
#df_book.explode()

#df_book.limit(5).select("id", "editor_array").show() #a test of uploading...

#df_author.printSchema()
#df_author.limit(5).show()
df_author = df_author \
  .withColumn("id" ,
              df_author["id"]
              .cast(StringType()))   \
  .withColumn("name"  ,
              df_author["name"]
              .cast(StringType())) \
  .withColumn("citationscore",
              df_author["citationscore"]
              .cast(FloatType()))    \
#email is already a string
#df_author.printSchema() 
#df_author.limit(5).show()

#the bridge tables have already id string types

root
 |-- editors_array: array (nullable = true)
 |    |-- element: string (containsNull = false)

+--------------------+
|       editors_array|
+--------------------+
|                null|
|                null|
|              [5180]|
|       [5052,  5180]|
|[5182,  5187,  51...|
|              [4881]|
|       [5190,  5189]|
|              [5191]|
|       [5192,  5193]|
|              [5194]|
|              [5195]|
|              [5196]|
|       [5198,  5197]|
|              [5199]|
|              [4881]|
|              [5200]|
|              [5201]|
|              [5202]|
|[5203,  5204,  5205]|
|[5208,  5206,  52...|
+--------------------+
only showing top 20 rows

root
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- isbn: string (nullable = true)
 |-- citationScore: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- title: string (nullable = true)
 |-- editor_array: string (nullable = true)

root
 |--